### Landsat por municipio 


In [1]:
SHP_PATH = r"D:\Arq-Azzoni\UrbanSprawl\Bases_dados\Shapes\BR_Municipios_2024\BR_Municipios_2024.shp"

Grupo de cidades

In [ ]:
import time
import ee
import geemap
import geopandas as gpd

LANDSAT = {
    "LT05": "LANDSAT/LT05/C02/T1_L2",
    "LE07": "LANDSAT/LE07/C02/T1_L2",
    "LC08": "LANDSAT/LC08/C02/T1_L2",
    "LC09": "LANDSAT/LC09/C02/T1_L2",
}

YEARS = [2000, 2010, 2022]
QUARTERS = [1, 2, 3, 4]

PROJECT = "satexport"
EXPORT_FOLDER = "gee_exports"
EXPORT_SCALE = 30
MAX_PIXELS = int(1e11)
POLL_SECONDS = 20
EXPORT_PREFIX = "landsat_quarterly_jundiai"


CD_MUN_LIST = [
    "3525904",  # Jundiaí
    "3523404",  # Itatiba
    "3527306",  # Louveira
    "3556701",  # Vinhedo
    "3525201",  # Jarinu
    "3509601",  # Campo Limpo Paulista
    "3556503",  # Várzea Paulista
    "3524006",  # Itupeva
    "3508405",  # Cabreúva
    "3509205",  # Cajamar
    "3516408",  # Franco da Rocha
    "3539103",  # Pirapora do Bom Jesus
]


def init_ee(project: str) -> None:
    try:
        ee.Initialize(project=project)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=project)


def load_municipios_from_shp(shp_path: str, cd_mun_list):
    gdf = gpd.read_file(shp_path)

    if gdf.empty:
        raise ValueError("O shapefile foi lido, mas está vazio.")

    field_name = "CD_MUN"
    if field_name not in gdf.columns:
        raise ValueError(
            f"Campo {field_name} não encontrado. Campos disponíveis: {gdf.columns.tolist()}"
        )

    gdf = gdf[gdf.geometry.notnull()].copy()

    invalid_count = int((~gdf.is_valid).sum())
    print("Geometrias inválidas antes da correção:", invalid_count)
    if invalid_count > 0:
        gdf["geometry"] = gdf.geometry.buffer(0)

    gdf[field_name] = gdf[field_name].astype(str).str.strip()
    cd_mun_list = [str(x).strip() for x in cd_mun_list]

    gdf_sel = gdf[gdf[field_name].isin(cd_mun_list)].copy()

    encontrados = sorted(gdf_sel[field_name].unique().tolist())
    faltando = sorted(set(cd_mun_list) - set(encontrados))

    print("Municípios encontrados:", encontrados)
    print("Municípios faltando:", faltando)

    if gdf_sel.empty:
        raise ValueError("Nenhum município encontrado para a lista informada.")

    if faltando:
        raise ValueError(f"Os seguintes CD_MUN não foram encontrados: {faltando}")

    target_gdf = gdf_sel.dissolve()

    municipios_fc = geemap.gdf_to_ee(gdf_sel)
    target_geom = geemap.gdf_to_ee(target_gdf).geometry()
    center = target_geom.centroid(1)

    return municipios_fc, target_geom, center


def build_search_buffer(target_geom: ee.Geometry, center: ee.Geometry) -> ee.Geometry:
    bounds = target_geom.bounds(1)
    coords = ee.List(bounds.coordinates().get(0))

    def vertex_to_feature(coord):
        pt = ee.Geometry.Point(coord)
        dist = center.distance(pt, 1)
        return ee.Feature(pt, {"dist": dist})

    vertices_fc = ee.FeatureCollection(coords.map(vertex_to_feature))
    max_dist = ee.Number(vertices_fc.aggregate_max("dist"))

    radius = max_dist.multiply(1.05)
    search_geom = center.buffer(radius, 1)

    contains_all = search_geom.contains(target_geom, ee.ErrorMargin(1)).getInfo()
    print(f"Buffer cobre todos os municípios? {contains_all}")
    print(f"Raio do buffer (m): {radius.getInfo():.2f}")

    if not contains_all:
        raise RuntimeError("O buffer calculado não cobre todos os municípios.")

    return search_geom


def sensors_for_year(year: int) -> list[str]:
    if year <= 2012:
        return ["LT05", "LE07"] if year >= 1999 else ["LT05"]
    if year <= 2021:
        return ["LC08"]
    return ["LC08", "LC09"]


def apply_scale_factors(img: ee.Image) -> ee.Image:
    optical = img.select("SR_B.").multiply(0.0000275).add(-0.2)
    thermal = img.select("ST_B.*").multiply(0.00341802).add(149.0)
    return img.addBands(optical, overwrite=True).addBands(thermal, overwrite=True)


def mask_clouds(img: ee.Image, sensor: str) -> ee.Image:
    qa = img.select("QA_PIXEL")

    mask = (
        qa.bitwiseAnd(1 << 1).eq(0)
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
        .And(qa.bitwiseAnd(1 << 5).eq(0))
    )

    if sensor in ("LC08", "LC09"):
        mask = mask.And(qa.bitwiseAnd(1 << 2).eq(0))

    sat = img.select("QA_RADSAT").eq(0)
    return img.updateMask(mask.And(sat))


def add_indices_temp(img: ee.Image, sensor: str) -> ee.Image:
    img = apply_scale_factors(img)
    img = mask_clouds(img, sensor)

    if sensor in ("LT05", "LE07"):
        red = img.select("SR_B3")
        nir = img.select("SR_B4")
        swir1 = img.select("SR_B5")
        temp_k = img.select("ST_B6")
    else:
        red = img.select("SR_B4")
        nir = img.select("SR_B5")
        swir1 = img.select("SR_B6")
        temp_k = img.select("ST_B10")

    ndvi = nir.subtract(red).divide(nir.add(red)).rename("ndvi")
    ndbi = swir1.subtract(nir).divide(swir1.add(nir)).rename("ndbi")
    temp_c = temp_k.subtract(273.15).rename("tempC")

    return ee.Image.cat([ndvi, ndbi, temp_c]).copyProperties(img, img.propertyNames())


def base_collection(collection_id: str, search_geom: ee.Geometry, start: ee.Date, end: ee.Date) -> ee.ImageCollection:
    return (
        ee.ImageCollection(collection_id)
        .filterBounds(search_geom)
        .filterDate(start, end)
        .filter(ee.Filter.eq("PROCESSING_LEVEL", "L2SP"))
        .filter(ee.Filter.lte("CLOUD_COVER_LAND", 80))
    )


def quarterly_collection(year: int, quarter: int, search_geom: ee.Geometry):
    start_month = (quarter - 1) * 3 + 1
    start = ee.Date.fromYMD(year, start_month, 1)
    end = start.advance(3, "month")

    merged = None
    for sensor in sensors_for_year(year):
        col = base_collection(LANDSAT[sensor], search_geom, start, end).map(
            lambda i, s=sensor: add_indices_temp(i, s)
        )
        merged = col if merged is None else merged.merge(col)

    if merged is None:
        merged = ee.ImageCollection([])

    return merged, start, end


def empty_output_image(region: ee.Geometry) -> ee.Image:
    ndvi_mean = ee.Image.constant(-9999).rename("ndvi_mean").clip(region).toFloat()
    ndvi_median = ee.Image.constant(-9999).rename("ndvi_median").clip(region).toFloat()

    ndbi_mean = ee.Image.constant(-9999).rename("ndbi_mean").clip(region).toFloat()
    ndbi_median = ee.Image.constant(-9999).rename("ndbi_median").clip(region).toFloat()

    temp_mean = ee.Image.constant(-9999).rename("tempC_mean").clip(region).toFloat()
    temp_median = ee.Image.constant(-9999).rename("tempC_median").clip(region).toFloat()
    temp_min = ee.Image.constant(-9999).rename("tempC_min").clip(region).toFloat()
    temp_max = ee.Image.constant(-9999).rename("tempC_max").clip(region).toFloat()

    n_obs = ee.Image.constant(0).rename("n_obs").clip(region).toFloat()
    mask_valid = ee.Image.constant(0).rename("mask_valid").clip(region).toFloat()

    return ee.Image.cat([
        ndvi_mean, ndvi_median,
        ndbi_mean, ndbi_median,
        temp_mean, temp_median, temp_min, temp_max,
        n_obs, mask_valid
    ]).toFloat()


def quarterly_composite(year: int, quarter: int, search_geom: ee.Geometry, target_geom: ee.Geometry, cd_list: list[str]):
    col, start, end = quarterly_collection(year, quarter, search_geom)
    count = int(col.size().getInfo())

    if count == 0:
        out = empty_output_image(target_geom)
    else:
        ndvi_mean = col.select("ndvi").mean().rename("ndvi_mean").toFloat()
        ndvi_median = col.select("ndvi").median().rename("ndvi_median").toFloat()

        ndbi_mean = col.select("ndbi").mean().rename("ndbi_mean").toFloat()
        ndbi_median = col.select("ndbi").median().rename("ndbi_median").toFloat()

        temp_mean = col.select("tempC").mean().rename("tempC_mean").toFloat()
        temp_median = col.select("tempC").median().rename("tempC_median").toFloat()
        temp_min = col.select("tempC").min().rename("tempC_min").toFloat()
        temp_max = col.select("tempC").max().rename("tempC_max").toFloat()

        n_obs = col.select("ndvi").count().rename("n_obs").toFloat()
        mask_valid = n_obs.gt(0).rename("mask_valid").toFloat()

        out = ee.Image.cat([
            ndvi_mean, ndvi_median,
            ndbi_mean, ndbi_median,
            temp_mean, temp_median, temp_min, temp_max,
            n_obs, mask_valid
        ]).clip(target_geom).toFloat()

    out = out.set({
        "year": year,
        "quarter": quarter,
        "cd_mun_list": ",".join(cd_list),
        "n_municipios": len(cd_list),
        "start_date": start.format("YYYY-MM-dd"),
        "end_date": end.advance(-1, "day").format("YYYY-MM-dd"),
        "sensors": ",".join(sensors_for_year(year)),
        "scene_count": count,
        "system:time_start": start.millis(),
    })

    out = out.toFloat()
    return out, count


def export_quarter(year: int, quarter: int, img: ee.Image, target_geom: ee.Geometry, cd_list: list[str]):
    group_id = f"grupo_{len(cd_list)}mun"
    name = f"{EXPORT_PREFIX}_{group_id}_{year}Q{quarter}"

    task = ee.batch.Export.image.toDrive(
        image=img,
        description=name,
        folder=EXPORT_FOLDER,
        fileNamePrefix=name,
        region=target_geom,
        scale=EXPORT_SCALE,
        maxPixels=MAX_PIXELS,
        fileFormat="GeoTIFF",
        skipEmptyTiles=True,
    )
    task.start()
    return task, name


def wait_for_task(task: ee.batch.Task, poll_seconds: int = POLL_SECONDS) -> str:
    while True:
        status = task.status()
        state = status.get("state", "UNKNOWN")
        print(f"Task {task.id}: {state}")

        if state == "COMPLETED":
            return state

        if state in {"FAILED", "CANCELLED", "CANCEL_REQUESTED"}:
            error_message = status.get("error_message", "Sem detalhes.")
            raise RuntimeError(f"Task {task.id} terminou com estado {state}: {error_message}")

        time.sleep(poll_seconds)


def process_one_quarter(year: int, quarter: int, search_geom: ee.Geometry, target_geom: ee.Geometry, cd_list: list[str]) -> None:
    print(f"\nProcessando municipios={cd_list} | {year} Q{quarter}")

    img, count = quarterly_composite(year, quarter, search_geom, target_geom, cd_list)
    print(f"Cenas encontradas: {count}")
    print(f"Sensores: {sensors_for_year(year)}")

    task, name = export_quarter(year, quarter, img, target_geom, cd_list)
    print(f"Export iniciado: {name} | task id: {task.id}")

    final_state = wait_for_task(task)
    print(f"Export finalizado: {name} | estado: {final_state}")


def main() -> None:
    init_ee(PROJECT)

    _, target_geom, center = load_municipios_from_shp(SHP_PATH, CD_MUN_LIST)
    search_geom = build_search_buffer(target_geom, center)

    for year in YEARS:
        for quarter in QUARTERS:
            process_one_quarter(year, quarter, search_geom, target_geom, CD_MUN_LIST)

    print("\nTodos os trimestres foram processados.")


if __name__ == "__main__":
    main()

Geometrias inválidas antes da correção: 1
Municípios encontrados: ['3508405', '3509205', '3509601', '3516408', '3523404', '3524006', '3525201', '3525904', '3527306', '3539103', '3556503', '3556701']
Municípios faltando: []
Buffer cobre todos os municípios? True
Raio do buffer (m): 43835.73

Processando municipios=['3525904', '3523404', '3527306', '3556701', '3525201', '3509601', '3556503', '3524006', '3508405', '3509205', '3516408', '3539103'] | 2000 Q1
Cenas encontradas: 10
Sensores: ['LT05', 'LE07']
Export iniciado: landsat_quarterly_jundiai_grupo_12mun_2000Q1 | task id: V4XA2IGFHDELUZCFHCZZGN3H
Task V4XA2IGFHDELUZCFHCZZGN3H: READY
Task V4XA2IGFHDELUZCFHCZZGN3H: RUNNING
Task V4XA2IGFHDELUZCFHCZZGN3H: RUNNING
Task V4XA2IGFHDELUZCFHCZZGN3H: RUNNING
Task V4XA2IGFHDELUZCFHCZZGN3H: RUNNING
Task V4XA2IGFHDELUZCFHCZZGN3H: RUNNING
Task V4XA2IGFHDELUZCFHCZZGN3H: RUNNING
Task V4XA2IGFHDELUZCFHCZZGN3H: RUNNING
Task V4XA2IGFHDELUZCFHCZZGN3H: RUNNING
Task V4XA2IGFHDELUZCFHCZZGN3H: RUNNING
Task V4

Uma cidade por vez

In [ ]:
import time
import ee

LANDSAT = {
    "LT05": "LANDSAT/LT05/C02/T1_L2",
    "LE07": "LANDSAT/LE07/C02/T1_L2",
    "LC08": "LANDSAT/LC08/C02/T1_L2",
    "LC09": "LANDSAT/LC09/C02/T1_L2",
}

YEARS = [2000, 2010, 2022]
QUARTERS = [1, 2, 3, 4]

PROJECT = "satexport"
EXPORT_FOLDER = "gee_exports"
EXPORT_SCALE = 30
MAX_PIXELS = int(1e11)
POLL_SECONDS = 20
EXPORT_PREFIX = "landsat_quarterly"

# AOI de exemplo
CENTER_LON = -46.8842
CENTER_LAT = -23.2157
BUFFER_METERS = 15000


def init_ee(project: str) -> None:
    try:
        ee.Initialize(project=project)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=project)


def build_aoi() -> ee.Geometry:
    center = ee.Geometry.Point([CENTER_LON, CENTER_LAT])
    return center.buffer(BUFFER_METERS).bounds()


def sensors_for_year(year: int) -> list[str]:
    if year <= 2012:
        return ["LT05", "LE07"] if year >= 1999 else ["LT05"]
    if year <= 2021:
        return ["LC08"]
    return ["LC08", "LC09"]


def apply_scale_factors(img: ee.Image) -> ee.Image:
    optical = img.select("SR_B.").multiply(0.0000275).add(-0.2)
    thermal = img.select("ST_B.*").multiply(0.00341802).add(149.0)
    return img.addBands(optical, overwrite=True).addBands(thermal, overwrite=True)


def mask_clouds(img: ee.Image, sensor: str) -> ee.Image:
    qa = img.select("QA_PIXEL")

    mask = (
        qa.bitwiseAnd(1 << 1).eq(0)   # dilated cloud
        .And(qa.bitwiseAnd(1 << 3).eq(0))  # cloud
        .And(qa.bitwiseAnd(1 << 4).eq(0))  # cloud shadow
        .And(qa.bitwiseAnd(1 << 5).eq(0))  # snow
    )

    if sensor in ("LC08", "LC09"):
        mask = mask.And(qa.bitwiseAnd(1 << 2).eq(0))  # cirrus

    sat = img.select("QA_RADSAT").eq(0)
    return img.updateMask(mask.And(sat))


def add_indices_temp(img: ee.Image, sensor: str) -> ee.Image:
    img = apply_scale_factors(img)
    img = mask_clouds(img, sensor)

    if sensor in ("LT05", "LE07"):
        red = img.select("SR_B3")
        nir = img.select("SR_B4")
        swir1 = img.select("SR_B5")
        temp_k = img.select("ST_B6")
    else:
        red = img.select("SR_B4")
        nir = img.select("SR_B5")
        swir1 = img.select("SR_B6")
        temp_k = img.select("ST_B10")

    ndvi = nir.subtract(red).divide(nir.add(red)).rename("ndvi")
    ndbi = swir1.subtract(nir).divide(swir1.add(nir)).rename("ndbi")
    temp_c = temp_k.subtract(273.15).rename("tempC")

    return ee.Image.cat([ndvi, ndbi, temp_c]).copyProperties(img, img.propertyNames())


def base_collection(collection_id: str, region: ee.Geometry, start: ee.Date, end: ee.Date) -> ee.ImageCollection:
    return (
        ee.ImageCollection(collection_id)
        .filterBounds(region)
        .filterDate(start, end)
        .filter(ee.Filter.eq("PROCESSING_LEVEL", "L2SP"))
        .filter(ee.Filter.lte("CLOUD_COVER_LAND", 80))
    )


def quarterly_collection(year: int, quarter: int, region: ee.Geometry) -> tuple[ee.ImageCollection, ee.Date, ee.Date]:
    start_month = (quarter - 1) * 3 + 1
    start = ee.Date.fromYMD(year, start_month, 1)
    end = start.advance(3, "month")

    merged = None
    for sensor in sensors_for_year(year):
        col = base_collection(LANDSAT[sensor], region, start, end).map(
            lambda i, s=sensor: add_indices_temp(i, s)
        )
        merged = col if merged is None else merged.merge(col)

    if merged is None:
        merged = ee.ImageCollection([])

    return merged, start, end


def empty_output_image(region: ee.Geometry) -> ee.Image:
    ndvi_mean = ee.Image.constant(-9999).rename("ndvi_mean").clip(region).toFloat()
    ndvi_median = ee.Image.constant(-9999).rename("ndvi_median").clip(region).toFloat()

    ndbi_mean = ee.Image.constant(-9999).rename("ndbi_mean").clip(region).toFloat()
    ndbi_median = ee.Image.constant(-9999).rename("ndbi_median").clip(region).toFloat()

    temp_mean = ee.Image.constant(-9999).rename("tempC_mean").clip(region).toFloat()
    temp_median = ee.Image.constant(-9999).rename("tempC_median").clip(region).toFloat()
    temp_min = ee.Image.constant(-9999).rename("tempC_min").clip(region).toFloat()
    temp_max = ee.Image.constant(-9999).rename("tempC_max").clip(region).toFloat()

    n_obs = ee.Image.constant(0).rename("n_obs").clip(region).toInt16()
    mask_valid = ee.Image.constant(0).rename("mask_valid").clip(region).toByte()

    return ee.Image.cat([
        ndvi_mean, ndvi_median,
        ndbi_mean, ndbi_median,
        temp_mean, temp_median, temp_min, temp_max,
        n_obs, mask_valid
    ])


def quarterly_composite(year: int, quarter: int, region: ee.Geometry) -> tuple[ee.Image, int]:
    col, start, end = quarterly_collection(year, quarter, region)
    count = int(col.size().getInfo())

    if count == 0:
        out = empty_output_image(region)
    else:
        ndvi_mean = col.select("ndvi").mean().rename("ndvi_mean").toFloat()
        ndvi_median = col.select("ndvi").median().rename("ndvi_median").toFloat()

        ndbi_mean = col.select("ndbi").mean().rename("ndbi_mean").toFloat()
        ndbi_median = col.select("ndbi").median().rename("ndbi_median").toFloat()

        temp_mean = col.select("tempC").mean().rename("tempC_mean").toFloat()
        temp_median = col.select("tempC").median().rename("tempC_median").toFloat()
        temp_min = col.select("tempC").min().rename("tempC_min").toFloat()
        temp_max = col.select("tempC").max().rename("tempC_max").toFloat()

        n_obs = col.select("ndvi").count().rename("n_obs").toInt16()
        mask_valid = n_obs.gt(0).rename("mask_valid").toByte()

        out = ee.Image.cat([
            ndvi_mean, ndvi_median,
            ndbi_mean, ndbi_median,
            temp_mean, temp_median, temp_min, temp_max,
            n_obs, mask_valid
        ]).clip(region)

    out = out.set({
        "year": year,
        "quarter": quarter,
        "start_date": start.format("YYYY-MM-dd"),
        "end_date": end.advance(-1, "day").format("YYYY-MM-dd"),
        "sensors": ",".join(sensors_for_year(year)),
        "scene_count": count,
        "system:time_start": start.millis(),
    })

    return out, count


def export_quarter(year: int, quarter: int, img: ee.Image, region: ee.Geometry) -> tuple[ee.batch.Task, str]:
    name = f"{EXPORT_PREFIX}_{year}Q{quarter}"

    task = ee.batch.Export.image.toDrive(
        image=img,
        description=name,
        folder=EXPORT_FOLDER,
        fileNamePrefix=name,
        region=region,
        scale=EXPORT_SCALE,
        maxPixels=MAX_PIXELS,
    )
    task.start()
    return task, name


def wait_for_task(task: ee.batch.Task, poll_seconds: int = POLL_SECONDS) -> str:
    while True:
        status = task.status()
        state = status.get("state", "UNKNOWN")
        print(f"Task {task.id}: {state}")

        if state == "COMPLETED":
            return state

        if state in {"FAILED", "CANCELLED", "CANCEL_REQUESTED"}:
            error_message = status.get("error_message", "Sem detalhes.")
            raise RuntimeError(f"Task {task.id} terminou com estado {state}: {error_message}")

        time.sleep(poll_seconds)


def process_one_quarter(year: int, quarter: int, region: ee.Geometry) -> None:
    print(f"\nProcessando {year} Q{quarter} ...")

    img, count = quarterly_composite(year, quarter, region)
    print(f"Cenas encontradas: {count}")
    print(f"Sensores: {sensors_for_year(year)}")

    task, name = export_quarter(year, quarter, img, region)
    print(f"Export iniciado: {name} | task id: {task.id}")

    final_state = wait_for_task(task)
    print(f"Export finalizado: {name} | estado: {final_state}")


def main() -> None:
    init_ee(PROJECT)
    aoi = build_aoi()

    for year in YEARS:
        for quarter in QUARTERS:
            process_one_quarter(year, quarter, aoi)

    print("\nTodos os trimestres foram processados.")


if __name__ == "__main__":
    main()

Por Buffer

In [1]:
import ee

LANDSAT = {
    "LT05": "LANDSAT/LT05/C02/T1_L2",
    "LE07": "LANDSAT/LE07/C02/T1_L2",
    "LC08": "LANDSAT/LC08/C02/T1_L2",
    "LC09": "LANDSAT/LC09/C02/T1_L2",
}

YEARS = [2000, 2010, 2022]
QUARTERS = [1, 2, 3, 4]

try:
    ee.Initialize(project="satexport")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="satexport")

# Example AOI
center = ee.Geometry.Point([-46.8842, -23.2157])
aoi = center.buffer(15000).bounds()

def sensors_for_year(year):
    if year <= 2012:
        # Landsat 5 available to 2012; Landsat 7 from 1999 onward
        if year >= 1999:
            return ["LT05", "LE07"]
        return ["LT05"]
    elif year <= 2021:
        return ["LC08"]
    else:
        return ["LC08", "LC09"]

def apply_scale_factors(img):
    optical = img.select("SR_B.").multiply(0.0000275).add(-0.2)
    thermal = img.select("ST_B.*").multiply(0.00341802).add(149.0)
    return img.addBands(optical, overwrite=True).addBands(thermal, overwrite=True)

def mask_clouds(img):
    qa = img.select("QA_PIXEL")

    # Common QA bits across C2 L2
    clear = qa.bitwiseAnd(1 << 6).neq(0)
    not_dilated = qa.bitwiseAnd(1 << 1).eq(0)
    not_cloud = qa.bitwiseAnd(1 << 3).eq(0)
    not_shadow = qa.bitwiseAnd(1 << 4).eq(0)
    not_snow = qa.bitwiseAnd(1 << 5).eq(0)

    mask = clear.And(not_dilated).And(not_cloud).And(not_shadow).And(not_snow)

    # High-confidence cirrus exists for Landsat 8/9 C2 L2.
    has_cirrus = img.bandNames().contains("QA_PIXEL")
    cirrus_ok = qa.bitwiseAnd(1 << 2).eq(0)
    mask = mask.And(cirrus_ok)

    sat = img.select("QA_RADSAT").eq(0)
    return img.updateMask(mask.And(sat))

def masked_constant(name):
    return ee.Image.constant(0).rename(name).updateMask(ee.Image.constant(0))

def add_ndvi_temp(img, sensor):
    img = apply_scale_factors(img)
    img = mask_clouds(img)

    if sensor in ("LT05", "LE07"):
        red = img.select("SR_B3")
        nir = img.select("SR_B4")
        st_band = "ST_B6"
    else:
        red = img.select("SR_B4")
        nir = img.select("SR_B5")
        st_band = "ST_B10"

    ndvi = nir.subtract(red).divide(nir.add(red)).rename("ndvi")

    # Keep only scenes with usable surface temperature
    tempK = ee.Image(
        ee.Algorithms.If(
            img.get("PROCESSING_LEVEL"),
            img.select(st_band),
            masked_constant(st_band)
        )
    )

    # More robust option: filter collection to L2SP before mapping
    tempK = ee.Image(
        ee.Algorithms.If(
            ee.String(img.get("PROCESSING_LEVEL")).compareTo("L2SP").eq(0),
            img.select(st_band),
            masked_constant(st_band)
        )
    )

    tempC = tempK.subtract(273.15).rename("tempC")
    return ee.Image.cat([ndvi, tempC]).copyProperties(img, img.propertyNames())

def base_collection(collection_id, aoi, start, end):
    return (
        ee.ImageCollection(collection_id)
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.eq("PROCESSING_LEVEL", "L2SP"))  # important for temperature
        .filter(ee.Filter.lte("CLOUD_COVER_LAND", 80))
    )

def quarterly_collection(year, quarter, aoi):
    start_month = (quarter - 1) * 3 + 1
    start = ee.Date.fromYMD(year, start_month, 1)
    end = start.advance(3, "month")

    merged = None
    for sensor in sensors_for_year(year):
        col = base_collection(LANDSAT[sensor], aoi, start, end).map(
            lambda i, s=sensor: add_ndvi_temp(i, s)
        )
        merged = col if merged is None else merged.merge(col)

    return merged, start, end

def quarterly_composite(year, quarter, aoi):
    col, start, end = quarterly_collection(year, quarter, aoi)

    ndvi = col.select("ndvi").median().rename("ndvi")
    tempC = col.select("tempC").median().rename("tempC")
    n_obs = col.select("ndvi").count().rename("n_obs")

    out = ee.Image.cat([ndvi, tempC, n_obs]).clip(aoi).toFloat()

    return out.set({
        "year": year,
        "quarter": quarter,
        "start_date": start.format("YYYY-MM-dd"),
        "end_date": end.advance(-1, "day").format("YYYY-MM-dd"),
        "sensors": ",".join(sensors_for_year(year)),
        "system:time_start": start.millis(),
    }), col.size()

def export_quarter(year, quarter, img):
    name = f"jundiai_{year}Q{quarter}_ndvi_temp"

    task = ee.batch.Export.image.toDrive(
        image=img,
        description=name,
        folder="gee_exports",
        fileNamePrefix=name,
        region=aoi,
        scale=30,
        maxPixels=int(1e11),
    )
    task.start()
    return task

tasks = []
for year in YEARS:
    for q in QUARTERS:
        img, count = quarterly_composite(year, q, aoi)
        print(f"{year} Q{q} | scenes: {count.getInfo()} | sensors: {sensors_for_year(year)}")
        task = export_quarter(year, q, img)
        tasks.append(task)
        print("Started:", task.id)

print("Total tasks:", len(tasks))

2000 Q1 | scenes: 6 | sensors: ['LT05', 'LE07']
Started: XVF6UL6BFMI6QYMQL4GH4LFL
2000 Q2 | scenes: 8 | sensors: ['LT05', 'LE07']
Started: MPAVHIJ2L4ALCLSUKSULERCZ
2000 Q3 | scenes: 8 | sensors: ['LT05', 'LE07']
Started: 3FDA3VI3YV4Q3PVSN2O5TH7G
2000 Q4 | scenes: 8 | sensors: ['LT05', 'LE07']
Started: X3WB6LMFEK2BCPHQMZHEUECS
2010 Q1 | scenes: 7 | sensors: ['LT05', 'LE07']
Started: YPM2UG5HWL5HJ2AKVQVJPFTV
2010 Q2 | scenes: 8 | sensors: ['LT05', 'LE07']
Started: VTVF3MLNOXCCZM33L56BRGWR
2010 Q3 | scenes: 5 | sensors: ['LT05', 'LE07']
Started: LW6F2ZECM4OC6X4AT73C5ADE
2010 Q4 | scenes: 6 | sensors: ['LT05', 'LE07']
Started: 6VE24DXRQPCADN7NQOFONWEY
2022 Q1 | scenes: 8 | sensors: ['LC08', 'LC09']
Started: RKS7CMVV3FP7MMOBR54DTG7I
2022 Q2 | scenes: 12 | sensors: ['LC08', 'LC09']
Started: MWPP2S5XGPZBFQXFO4POP6G7
2022 Q3 | scenes: 8 | sensors: ['LC08', 'LC09']
Started: 4HUCFY3XCO5MZBVC7CWWO4IZ
2022 Q4 | scenes: 11 | sensors: ['LC08', 'LC09']
Started: E3VGXOH5KIDHRRNSFQX4UVBQ
Total tasks: 1

Agregar por ano 

In [4]:
from pathlib import Path
import re
import numpy as np
import rasterio
import warnings

# =====================================
# 1) PATHS
# =====================================
input_folder = Path(r"D:\Users\ivan.cavalcanti\Documents\Projects\extract_landsat\outputs\drive")

output_folder = input_folder / "annual"
output_folder.mkdir(exist_ok=True)

# Match files like: jundiai_2000Q1_ndvi_temp.tif
pattern = re.compile(
    r"^(?P<city>.+?)_(?P<year>\d{4})Q(?P<quarter>[1-4])_ndvi_temp\.tif$",
    re.IGNORECASE
)

# =====================================
# 2) GROUP FILES BY (CITY, YEAR)
# =====================================
files_by_year = {}

for fp in input_folder.glob("*.tif"):
    m = pattern.match(fp.name)
    if not m:
        continue

    city = m.group("city")
    year = int(m.group("year"))
    quarter = int(m.group("quarter"))

    files_by_year.setdefault((city, year), {})[quarter] = fp

if not files_by_year:
    raise ValueError(f"Nenhum arquivo compatível encontrado em: {input_folder}")

# =====================================
# 3) PROCESS EACH YEAR
# =====================================
for (city, year), quarter_files in sorted(files_by_year.items()):
    print(f"\nProcessing {city} - {year}")

    missing = [q for q in [1, 2, 3, 4] if q not in quarter_files]
    if missing:
        print(f"Skipping {city} {year}: missing quarters {missing}")
        continue

    ordered_files = [quarter_files[q] for q in [1, 2, 3, 4]]

    arrays = []
    profile = None
    nodata = None
    ref_shape = None
    ref_crs = None
    ref_transform = None
    ref_count = None
    band_descriptions = None

    for fp in ordered_files:
        with rasterio.open(fp) as src:
            data = src.read().astype("float32")  # (bands, rows, cols)

            if profile is None:
                profile = src.profile.copy()
                nodata = src.nodata
                ref_shape = data.shape
                ref_crs = src.crs
                ref_transform = src.transform
                ref_count = src.count

                # Try to preserve original band descriptions
                band_descriptions = list(src.descriptions)
                if not band_descriptions or all(d is None for d in band_descriptions):
                    # fallback if missing
                    fallback = ["ndvi", "tempC", "n_obs"]
                    band_descriptions = fallback[:src.count]
            else:
                if data.shape != ref_shape:
                    raise ValueError(
                        f"Shape mismatch in {fp.name}: {data.shape} vs {ref_shape}"
                    )
                if src.crs != ref_crs:
                    raise ValueError(
                        f"CRS mismatch in {fp.name}: {src.crs} vs {ref_crs}"
                    )
                if src.transform != ref_transform:
                    raise ValueError(f"Transform mismatch in {fp.name}")
                if src.count != ref_count:
                    raise ValueError(
                        f"Band count mismatch in {fp.name}: {src.count} vs {ref_count}"
                    )

            if nodata is not None:
                data[data == nodata] = np.nan

            arrays.append(data)

    # Stack: (quarters, bands, rows, cols)
    stack = np.stack(arrays, axis=0)

    # Mean across quarters
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        annual_mean = np.nanmean(stack, axis=0).astype("float32")

    out_nodata = nodata if nodata is not None else -9999.0
    annual_out = np.where(np.isnan(annual_mean), out_nodata, annual_mean).astype("float32")

    profile.update(
        dtype="float32",
        count=annual_out.shape[0],
        nodata=out_nodata,
        compress="lzw"
    )

    out_fp = output_folder / f"{city}_{year}_annual_mean_ndvi_temp.tif"

    with rasterio.open(out_fp, "w", **profile) as dst:
        dst.write(annual_out)

        for i in range(min(dst.count, len(band_descriptions))):
            if band_descriptions[i] is not None:
                dst.set_band_description(i + 1, band_descriptions[i])

    print(f"Saved: {out_fp}")

print("\nDone.")


Processing jundiai - 2000
Saved: D:\Users\ivan.cavalcanti\Documents\Projects\extract_landsat\outputs\drive\annual\jundiai_2000_annual_mean_ndvi_temp.tif

Processing jundiai - 2010
Saved: D:\Users\ivan.cavalcanti\Documents\Projects\extract_landsat\outputs\drive\annual\jundiai_2010_annual_mean_ndvi_temp.tif

Processing jundiai - 2022
Saved: D:\Users\ivan.cavalcanti\Documents\Projects\extract_landsat\outputs\drive\annual\jundiai_2022_annual_mean_ndvi_temp.tif

Done.


In [5]:
with rasterio.open(output_folder / "jundiai_2000_annual_mean_ndvi_temp.tif") as src:
    print(src.descriptions)

('ndvi', 'tempC', 'n_obs')
